#Semantic Analysis Tools
Taking in stream of words in the form of pkl file. Applying cosine similarity, then placing it on a table.

Disclaimer: We utilize tools from HistWords, Gensim, and GPT. Here are links to the tools for ease of access. Furthermore, we utilize other generic python libraries as imported below.

https://radimrehurek.com/gensim/auto_examples/index.html#documentation
https://github.com/williamleif/histwords
https://developers.openai.com/api/docs/models/gpt-5-nano

##Setup

In [1]:
!mkdir embeddings
!cd embeddings
#!curl -o eng-fiction-all_sgns.zip http://snap.stanford.edu/historical_embeddings/eng-fiction-all_sgns.zip
#!unzip eng-fiction-all_sgns.zip
#!mv sgns eng-fiction-all_sgns

!curl -o eng-fiction-all.zip http://snap.stanford.edu/historical_embeddings/eng-fiction-all.zip
!unzip eng-fiction-all.zip


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 2503M  100 2503M    0     0  32.0M      0  0:01:18  0:01:18 --:--:-- 36.6M
Archive:  eng-fiction-all.zip
   creating: eng-fiction-all/
   creating: eng-fiction-all/netstats/
  inflating: eng-fiction-all/netstats/full-nstop_nproper-top10000.pkl  
   creating: eng-fiction-all/svd/
  inflating: eng-fiction-all/svd/1860-vocab.pkl  
  inflating: eng-fiction-all/svd/1900-vocab.pkl  
  inflating: eng-fiction-all/svd/1860-w.npy  
  inflating: eng-fiction-all/svd/1900-w.npy  
  inflating: eng-fiction-all/svd/1840-w.npy  
  inflating: eng-fiction-all/svd/1920-w.npy  
  inflating: eng-fiction-all/svd/1890-w.npy  
  inflating: eng-fiction-all/svd/1970-vocab.pkl  
  inflating: eng-fiction-all/svd/1810-vocab.pkl  
  inflating: eng-fiction-all/svd/1940-w.npy  
  inflating: eng-fiction-all/svd/1920-vocab.pkl  
  inflating: eng-fiction-all/svd

In [2]:
!pip install --upgrade gensim
!pip install openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 58.9 MB/s eta 0:00:00


In [3]:
import numpy as np
import pickle as p
from openai import OpenAI
from google.colab import userdata
from gensim.models import KeyedVectors
import pandas as pd
import random
from IPython.display import display, Markdown

Find minimum frequency words.

In [4]:
with open(f"/content/eng-fiction-all/word_lists/full-nstop_nproper.pkl", "rb") as file:
    clean_words_sorted = p.load(file, encoding="latin1")

with open(f"/content/eng-fiction-all/freqs.pkl", "rb") as file:
    freq_of_words = p.load(file, encoding="latin1")

with open(f"/content/eng-fiction-all/volstats/vols.pkl", "rb") as file:
    score_of_words = p.load(file, encoding="latin1")

In [5]:
selected_decades = [decade for decade in range(1950, 1990+10, 10)]

In [6]:
top_freq_words = clean_words_sorted[:10000]
sanitized_top_freq_words = top_freq_words

In [7]:
for word in score_of_words:
  for decade in selected_decades:
    if np.isnan(score_of_words[word][decade]):
      if word in sanitized_top_freq_words:
        sanitized_top_freq_words.remove(word)

In [8]:
def avg_shift_distance(key):
  cumulative_distance_shift = 0
  for decade in selected_decades:
    cumulative_distance_shift += np.arccos(np.clip(score_of_words[key][decade],-1, 1))
  avg_shift_distance = cumulative_distance_shift/len(selected_decades)
  return avg_shift_distance

In [9]:
shift_list = []

for word in sanitized_top_freq_words:
  shift_list.append([word, avg_shift_distance(word)])

shift_list = sorted(shift_list,key=lambda x: x[1], reverse=True)


In [10]:
percentile = 0.25

low_shift_words = shift_list[int(len(shift_list)-len(shift_list)*percentile):]
high_shift_words = shift_list[:int(len(shift_list)*percentile)]

In [11]:
seed = 1042
num_words = 50

random.seed(seed)
low_target_word = random.sample(low_shift_words, num_words)
high_target_word = random.sample(high_shift_words, num_words)

In [12]:
models = []
count = 0
for decade in selected_decades:
  model = [decade]
  vectors = np.load(f"/content/eng-fiction-all/sgns/{decade}-w.npy")
  with open(f"/content/eng-fiction-all/sgns/{decade}-vocab.pkl", "rb") as file:
    keys = p.load(file)

  model.append(keys)
  model.append(KeyedVectors(vector_size=vectors.shape[1]))

  model[2].add_vectors(keys, vectors)
  models.append(model)

##Model Testing

In [13]:
def sanitize_list(output_list, model):
  t10ChatGPT = []
  for word in output_list:
    if word in model[1] and np.mean(model[2][word]) != 0:
      t10ChatGPT.append(word)
  return t10ChatGPT

def cosine_similarity(target_word, generated_words, model):
  target_vector = model[2][target_word]

  generated_vectors = [model[2][word] for word in generated_words]

  cos_sim_list = model[2].cosine_similarities(target_vector, generated_vectors)

  return np.mean(cos_sim_list)

In [14]:
with open("word_generations_high.pkl", "rb") as file:
  word_generations_high = p.load(file)

In [15]:
cos_sim_high = {}
for target_word in word_generations_high:
  count_dict = {}
  for count in word_generations_high[target_word]:
    decade_dict = {}
    for decade in selected_decades:
      model = models[selected_decades.index(decade)]
      temp_dict = {}
      allowed_prompts = ["ahistorical", "modern", "historical", str(decade)]
      for prompt in allowed_prompts:
        raw_words = word_generations_high[target_word][count][prompt]
        generated_words = sanitize_list(raw_words, model)
        if len(generated_words) == 0:
          temp_dict[prompt] = np.nan
        else:
          temp_dict[prompt] = cosine_similarity(
            target_word,
            generated_words,
            model
          )
      decade_dict[decade] = temp_dict
    count_dict[count] = decade_dict
  cos_sim_high[target_word] = count_dict

In [16]:
rows = []
for target_word in cos_sim_high:
  for count in cos_sim_high[target_word]:
    for decade in cos_sim_high[target_word][count]:
      for prompt in cos_sim_high[target_word][count][decade]:
        avg_score = cos_sim_high[target_word][count][decade][prompt]
        rows.append({
          "target_word": target_word,
          "count": count,
          "decade": decade,
          "prompt": prompt,
          "avg_cos_sim": avg_score
        })

df_high = pd.DataFrame(rows)
df_high

df1 = df_high.pivot_table(
  index="prompt",
  columns="decade",
  values="avg_cos_sim",
  aggfunc="mean"
)

df_high["prompt"] = df_high["prompt"].astype(str)

df_high["prompt"] = df_high["prompt"].replace({
  "1950": "historical_decade",
  "1960": "historical_decade",
  "1970": "historical_decade",
  "1980": "historical_decade",
  "1990": "historical_decade"
})

table_high = df_high.pivot_table(
  index="prompt",
  columns="decade",
  values="avg_cos_sim",
  aggfunc="mean"
)

row_order = ["ahistorical", "modern", "historical", "historical_decade"]
table_high = table_high.loc[row_order]

table_high.round(4)

display(Markdown("#Semantic Neighbors' Average Cosine Similarity Value"))
display(table_high)

#Semantic Neighbors' Average Cosine Similarity Value

decade,1950,1960,1970,1980,1990
prompt,,,,,
ahistorical,0.183719,0.178866,0.192491,0.186725,0.188839
modern,0.177465,0.172726,0.183540,0.182222,0.182803
historical,0.185385,0.180042,0.191543,0.189651,0.189972
historical_decade,0.184677,0.177789,0.189280,0.185053,0.186862


In [17]:
delta_high = table_high.subtract(
  table_high.loc["ahistorical"],
  axis=1
)

delta_high = delta_high.drop(index="ahistorical")

delta_high.round(4)

display(Markdown("#Delta in Semantic Neighbors Average Cosine Similarity Value"))
display(delta_high)

#Delta in Semantic Neighbors Average Cosine Similarity Value

decade,1950,1960,1970,1980,1990
prompt,,,,,
modern,-0.006254,-0.006139,-0.008951,-0.004503,-0.006036
historical,0.001666,0.001177,-0.000948,0.002926,0.001132
historical_decade,0.000957,-0.001076,-0.003211,-0.001672,-0.001977


In [18]:
with open("cos_sim_high.pkl", "wb") as file:
  p.dump(cos_sim_high, file)